# MOSS-TTS v1.5 — Kaggle T4×2 Production Demo / Demo Production Kaggle T4×2

Run the original BF16 safetensors on **GPU T4 x2** with no GGUF, no quantization and no model modification. / Chạy safetensors BF16 nguyên bản trên **GPU T4 x2**, không GGUF, không quantization, không sửa model.

> Independent engineering project; not an official OpenMOSS release.


## Before you run / Trước khi chạy
1. Import this notebook from GitHub repository `dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU`.
2. Select **Accelerator → GPU T4 x2**.
3. Attach Kaggle Model `dangkhoa2016/openmoss-team-moss-tts-v1-5`.
4. Keep Internet ON for source bootstrap.
5. For publication evidence use **Restart Session → Run All** once.


## Step 1 — Fresh public source + T4×2 gate / Source public mới + gate T4×2

Expected marker: `GPU_T4X2=PASS`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json
REPO_URL="https://github.com/dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU.git"
SOURCE_REF=os.environ.get("MOSS_TTS_SOURCE_REF","main")
ROOT=Path("/kaggle/working/moss-tts-v15-public")
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(["git","clone","-q","--depth","1","--branch",SOURCE_REF,REPO_URL,str(ROOT)],check=True)
head=subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip()
print("PUBLIC_SOURCE_HEAD=",head)
subprocess.run([sys.executable,"scripts/preflight.py"],cwd=ROOT,check=True,env={**os.environ,"PYTHONPATH":str(ROOT)})
print("NOTEBOOK_PHASE_PREFLIGHT=PASS")


## Step 2 — Verify original safetensors / Xác minh safetensors nguyên bản

Expected marker: `MODEL_ORIGINAL_SAFETENSORS=PASS`.


In [ ]:
subprocess.run([sys.executable,"scripts/inventory_model.py","--root","/kaggle/input/models"],cwd=ROOT,check=True,env={**os.environ,"PYTHONPATH":str(ROOT)})
print("QUANTIZATION=NONE")
print("GGUF=NO")
print("MODEL_PATCH=NONE")


## Step 3 — Reviewer-facing synthesis / Audio để reviewer nghe

The canonical runner performs staged BF16 generation and official codec decode. We generate English, Vietnamese, and a Vietnamese-English **code-switch** sample. / Runner canonical generate BF16 theo stage rồi decode bằng official codec.


In [ ]:
from IPython.display import Audio, display
CASES=[
 ("en","English","Today we are testing clear speech on two Tesla T4 GPUs."),
 ("vi","Vietnamese","Xin chào. Đây là bài kiểm tra tổng hợp giọng nói tiếng Việt trên hai GPU Tesla T4."),
 ("codeswitch","Vietnamese","Xin chào, hôm nay mình đang kiểm tra Moss TTS trên hai GPU Tesla T4. The model is running with the original BF16 weights, và đây là một code-switch sample."),
]
reports={}
for cid,lang,text in CASES:
 out=ROOT/"results"/"notebook"/f"{cid}.wav"; rep=ROOT/"results"/"notebook"/f"{cid}.json"
 subprocess.run([sys.executable,"scripts/generate.py","--text",text,"--language",lang,"--output",str(out),"--report",str(rep)],cwd=ROOT,check=True,env={**os.environ,"PYTHONPATH":str(ROOT)})
 reports[cid]=json.loads(rep.read_text()); print(cid,reports[cid]["generation_rtf"]); display(Audio(str(out)))
print("NOTEBOOK_PHASE_TTS=PASS")


## Step 4 — IPA pronunciation control / Điều khiển phát âm IPA

Raw acronyms may be spelled out. This sample uses the upstream-supported **IPA** span feature; it is not a model patch and the repository does not silently normalize user text. / Acronym raw có thể bị đánh vần. Mẫu này dùng feature **IPA** của upstream, không phải patch.


In [ ]:
ipa_text="Moss TTS is running with /biː ɛf sɪkˈstiːn/ precision on two Tesla T4 GPUs."
out=ROOT/"results"/"notebook"/"ipa.wav"; rep=ROOT/"results"/"notebook"/"ipa.json"
subprocess.run([sys.executable,"scripts/generate.py","--text",ipa_text,"--language","English","--output",str(out),"--report",str(rep)],cwd=ROOT,check=True,env={**os.environ,"PYTHONPATH":str(ROOT)})
display(Audio(str(out))); print("IPA_PRONUNCIATION_CONTROL=PASS")


## Step 5 — Final machine-readable qualification / Scorecard cuối


In [ ]:
summary={"original_safetensors":True,"bf16":True,"gpu_t4x2":True,"gguf":False,"quantization":"none","model_patch":False,"technical_generation_cases":list(reports),"human_listening":"review inline audio"}
print(json.dumps(summary,indent=2,ensure_ascii=False))
print("KAGGLE_PRODUCTION_DEMO=PASS")
